# End-to-end one-tube salt-analysis test

This notebook runs the complete hardware-safe flow in phases: RAG planning, optional Qwen fallback, CPU text-card rendering, persistent Wan generation, and final card/video composition. Qwen is never resident at the same time as Wan.


In [ ]:
import json, os, subprocess, sys
from pathlib import Path
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
ROOT = Path("/kaggle/working")
REPOS = {
    "Wan2.2": ("https://github.com/abhinavk0006/Wan2.2.git", "3558b40"),
    "image-to-video-pipeline": ("https://github.com/abhinavk0006/image-to-video-pipeline.git", "7a940bc"),
    "Edu-video-gen-dataset": ("https://github.com/abhinavk0006/Edu-video-gen-dataset.git", "53b5222"),
}
for name, (url, commit) in REPOS.items():
    path = ROOT / name
    if not path.exists():
        subprocess.run(["git", "clone", url, str(path)], check=True)
    subprocess.run(["git", "fetch", "origin", "main"], cwd=path, check=True)
    subprocess.run(["git", "checkout", "--detach", commit], cwd=path, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "Edu-video-gen-dataset" / "requirements.txt"), "imageio-ffmpeg", "Pillow"], check=True)
dry = list(Path("/kaggle/input").rglob("salt_analysis_one_test_tube_dry_salt.jpeg"))
acid = list(Path("/kaggle/input").rglob("salt_analysis_one_test_tube_acidified_solution.jpeg"))
assert len(dry) == 1 and len(acid) == 1, "Add exactly the two required one-tube images to Kaggle input"
DRY_IMAGE, ACID_IMAGE = dry[0], acid[0]
print("Inputs:", DRY_IMAGE, ACID_IMAGE)


## 1. RAG plan, with Qwen fallback only on a retrieval miss

The RAG result is used for experiment identity and educational metadata. The controlled four-step one-tube fixture below supplies the exact visual states and prevents an automatically generated plan from adding extra vessels.

In [ ]:
rag = ROOT / "Edu-video-gen-dataset"
bundle_path = rag / "exports" / "salt_analysis_end_to_end_bundle.json"
query = "Class 12 qualitative salt analysis ammonium chloride chloride test silver nitrate ammonia"
result = subprocess.run([sys.executable, "scripts/build_rag_prompt_bundle.py", query, "--top-k", "1", "--output", str(bundle_path)], cwd=rag, capture_output=True, text=True)
if result.returncode:
    raise RuntimeError(result.stderr or result.stdout)
rag_bundle = json.loads(bundle_path.read_text(encoding="utf-8"))
assert rag_bundle.get("retrieved_experiments"), "RAG returned no experiment"
rag_record = rag_bundle["retrieved_experiments"][0]
print("RAG selected:", rag_record.get("title"), "score=", rag_record.get("retrieval_score"))
print("Observation fields:", all("observation" in p for p in rag_bundle.get("prompt_inputs", [])))
del result, rag_bundle
import gc
gc.collect()


In [ ]:
pipeline = ROOT / "image-to-video-pipeline"
fixture_dir = pipeline / "kaggle_experiments"
fixture_dir.mkdir(exist_ok=True)
negative = "extra test tubes, duplicate vessels, multiple droppers, crowded rack, unrelated apparatus, blurry, warped glass, text, watermark, camera shake"
def make_clip(name, image, before, action, after, procedure, observation, motion, chained=False):
    clip = {"name": name, "input_frame_path": str(image) if image else None, "state_before": before, "action": action, "state_after": after, "visual_priority": "high", "continuity_required": chained, "reference_mode": "chained" if chained else "independent", "needs_reference_image": not chained, "duration_seconds": 2.0, "procedure_text": procedure, "observation_text": observation, "prompt_bundle": {"motion_prompt": motion, "negative_prompt": negative}}
    if chained:
        clip.update({"reference_policy": "previous_state_keyframe", "reference_clip": "silver_nitrate_chloride_test", "handoff_policy": "final", "handoff_entity": "white silver chloride precipitate"})
    else:
        clip["reference_policy"] = "state_keyframe"
    return clip
clips = [
  make_clip("prepare_unknown_salt", DRY_IMAGE, "one dry test tube with white unknown salt", "add water and swirl gently until the salt dissolves", "clear unknown salt solution", "Add distilled water to the dry unknown salt and dissolve it.", "The white salt dissolves to form a clear solution.", "Use exactly one test tube containing dry white salt. Add water and swirl gently until the salt dissolves. Do not create another vessel."),
  make_clip("acidify_unknown_solution", ACID_IMAGE, "one test tube with clear unknown salt solution", "add dilute nitric acid carefully", "clear acidified unknown salt solution", "Acidify a fresh portion of the clear salt solution with dilute nitric acid.", "The solution remains clear and colorless after acidification.", "Use exactly one test tube containing clear salt solution. Add dilute nitric acid carefully and keep the solution clear. Do not create another vessel."),
  make_clip("silver_nitrate_chloride_test", ACID_IMAGE, "one test tube containing clear acidified unknown salt solution", "add dilute silver nitrate slowly and swirl once", "one thick white curdy silver chloride precipitate remains visible", "Add dilute silver nitrate solution to the acidified salt solution.", "A thick white curdy precipitate of silver chloride forms.", "Use exactly one test tube. Add silver nitrate slowly. One white curdy precipitate forms in that same tube. Create no second vessel."),
  make_clip("ammonia_dissolves_chloride_ppt", None, "the same one test tube with white silver chloride precipitate", "add dilute ammonia gradually", "the precipitate dissolves, leaving one clear colorless solution", "Add dilute ammonia solution to the silver chloride precipitate.", "The white precipitate dissolves in ammonia and the solution becomes clear.", "Continue with the same single test tube and visible white precipitate. Add ammonia gradually. The precipitate dissolves in place. Do not introduce another tube.", chained=True),
]
fixture = {"name": "One-Tube Chloride Salt Analysis", "subject": "Qualitative salt analysis", "educational_goal": "Prepare one salt solution, confirm chloride using silver nitrate, and verify dissolution in ammonia.", "rag_experiment": rag_record, "clips": clips, "conclusion_text": "The white silver chloride precipitate dissolves in ammonia, confirming chloride ions in the unknown salt."}
fixture_path = fixture_dir / "salt_analysis_one_tube_end_to_end.json"
fixture_path.write_text(json.dumps(fixture, indent=2), encoding="utf-8")
print(fixture_path)


## 2. Render deterministic text cards on CPU

Cards use JSON values directly; no LLM is used for card text. They are rendered to PNG and converted to same-format MP4 segments before final composition.

In [ ]:
from PIL import Image, ImageDraw, ImageFont
from imageio_ffmpeg import get_ffmpeg_exe
cards_dir = ROOT / "wan_salt_analysis_end_to_end" / "cards"
cards_dir.mkdir(parents=True, exist_ok=True)
font = ImageFont.load_default()
def render_card(path, heading, body):
    image = Image.new("RGB", (624, 624), "white")
    draw = ImageDraw.Draw(image)
    draw.text((42, 42), heading, fill="black", font=font)
    y = 100
    for line in body:
        words, current = line.split(), ""
        for word in words:
            trial = (current + " " + word).strip()
            if draw.textbbox((0, 0), trial, font=font)[2] > 540:
                draw.text((42, y), current, fill="black", font=font); y += 24; current = word
            else: current = trial
        if current: draw.text((42, y), current, fill="black", font=font); y += 42
    image.save(path)
def card_video(png, mp4):
    subprocess.run([get_ffmpeg_exe(), "-y", "-loop", "1", "-i", str(png), "-t", "1.5", "-r", "16", "-vf", "scale=624:624", "-c:v", "libx264", "-pix_fmt", "yuv420p", str(mp4)], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
card_paths = []
render_card(cards_dir / "00_title.png", "One-Tube Chloride Salt Analysis", [fixture["educational_goal"]]); card_paths.append(cards_dir / "00_title.png")
for i, item in enumerate(clips, 1):
    render_card(cards_dir / f"{i:02d}_procedure.png", f"Step {i}: Procedure", [item["procedure_text"]])
    render_card(cards_dir / f"{i:02d}_observation.png", "Observation", [item["observation_text"]])
    card_paths.extend([cards_dir / f"{i:02d}_procedure.png", cards_dir / f"{i:02d}_observation.png"])
render_card(cards_dir / "99_conclusion.png", "Conclusion", [fixture["conclusion_text"]]); card_paths.append(cards_dir / "99_conclusion.png")
print("Rendered", len(card_paths), "text cards")
del font
gc.collect()


## 3. Release planning resources, then run Wan persistently


In [ ]:
OUT = ROOT / "wan_salt_analysis_end_to_end"
subprocess.run([sys.executable, "main.py", "--knowledge-file", str(fixture_path), "--initial-image", str(DRY_IMAGE), "--output-dir", str(OUT), "--generator-script", "kaggle_wan_wrapper.py", "--generator-working-dir", str(pipeline), "--wan-repo-dir", str(ROOT / "Wan2.2"), "--wan-model-preset", "i2v-a14b", "--continuity-mode", "chain", "--wan-max-area", "345600", "--wan-fallback-area", "230400", "--wan-fallback-area", "172800", "--generator-arg=--lightning", "--generator-arg=--gpu0-memory-gib=5", "--generator-arg=--gpu1-memory-gib=11", "--generator-arg=--memory-telemetry"], cwd=pipeline, check=True)
print("Wan generation complete")


In [ ]:
from stitcher import stitch_clips
experiment_dir = OUT / "one_tube_chloride_salt_analysis"
video_paths = [experiment_dir / f"clip_{i}_{name}" / f"{name}.mp4" for i, name in enumerate(["prepare_unknown_salt", "acidify_unknown_solution", "silver_nitrate_chloride_test", "ammonia_dissolves_chloride_ppt"], 1)]
card_videos = []
for png in sorted(cards_dir.glob("*.png")):
    mp4 = png.with_suffix(".mp4"); card_video(png, mp4); card_videos.append(mp4)
ordered = [card_videos[0], card_videos[1], video_paths[0], card_videos[2], video_paths[1], card_videos[3], video_paths[2], card_videos[4], video_paths[3], card_videos[5]]
final = stitch_clips(ordered, OUT / "salt_analysis_end_to_end_final.mp4")
print("Final educational video:", final)
